# The team that runs its own experiments reports smaller effects

It is a real pattern and it has two possible explanations. Either their model reads are
optimistic — a provenance bias worth estimating and correcting — or the teams that bother to
run experiments are simply working on different treatments. The two are indistinguishable
unless somebody, somewhere, reports *both* reads of the same effect.

That is why `delta` in this model is identified only by dual-read contributors, and why asking
for it without them returns the prior back rather than a number. The alternative — comparing
model-only teams with experiment-only teams — would let "who runs experiments" masquerade as a
bias in how models read.

For one family with records `i = 1..k` grouped into effects `g(i)`:

    theta_g ~ N(mu, tau)
    y_i     ~ N(theta_g(i) + x_i·gamma + delta·m_i,  se_i)      (known se_i)

`mu` is the family mean on the experimental scale, `tau` the between-effect sd, `gamma` the
coefficients of *centered* moderators, and `delta` the provenance offset a model read
(`m_i = 1`) carries. By default an effect is attached to a **contributor**, so a contributor's
model read and experimental read share one `theta` — which is what identifies `delta`.

`pool_model` builds the `core.ModelSpec`; `pool` fits it through an `infer` backend. The default
parametrization integrates `theta` out exactly (the marginal form), which is what makes a
Laplace fit exact for fixed `tau` and well-posed for free `tau`; the effects are recovered draw
by draw from their conditional.

In [ ]:
import numpy as np

from axiom.core import Posterior, Spec, Unsupported, Verdict
from axiom.meta import (
    NO_DUAL_READ, Corpus, EffectShrinkage, ModeratorDesign, ParameterSummary, Pooled, PoolPriors,
    PoolResult, PoolSpec, StudyRecord, delta_identification, moderator_matrix, pool, pool_model,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import BLUE, ORANGE, caption, density, dumbbell, mark_x

enable();  # every axiom result renders itself from here on

## A simulated corpus with known `tau`

Twenty contributors, one experimental read each, true effects `theta_g ~ N(0.5, 0.3)`, each
observed with its own se. A moderator `follow_up` with a known coefficient of `0.05` per unit
(centered at its mean) enters the mean.

In [ ]:
MU, TAU, GAMMA = 0.5, 0.3, 0.05
rng = np.random.default_rng(7)
G = 20
se_true = rng.uniform(0.08, 0.3, G)
follow_up = rng.integers(2, 13, G).astype(float)
theta_true = rng.normal(MU, TAU, G)
y_obs = theta_true + GAMMA * (follow_up - follow_up.mean()) + se_true * rng.normal(size=G)

records = tuple(
    StudyRecord(study=f"s{g}", contributor=f"c{g}", quantity="elasticity", estimate=float(y_obs[g]),
                se=float(se_true[g]), read="experiment", family="fertilizer",
                moderators={"follow_up": float(follow_up[g])})
    for g in range(G)
)
corpus = Corpus(records=records, name="simulated")
print(len(corpus), "records |", len(corpus.contributors()), "contributors")

## `PoolPriors`, `PoolSpec`, `pool_model`

Every prior scale is explicit on `PoolPriors` (`mu ~ N(0, mu_scale)`, `tau ~ HalfNormal(tau_scale)`,
`gamma_j ~ N(0, gamma_scale)`, `delta ~ N(0, delta_scale)`); `tau_fixed` replaces the
half-normal by a point mass. `pool_model` returns the `ModelSpec` and the data dict the backend
sees. In the marginal form the likelihood's known per-row scale is an expression,
`scale_expr = sqrt(1 + tau² · tau2_load)`, rather than a parameter name.

In [ ]:
priors = PoolPriors(mu_scale=2.0, tau_scale=0.5, gamma_scale=0.5)
spec = PoolSpec(family="fertilizer", moderators=("follow_up",), priors=priors, mass=0.9, definition="eti")
built = pool_model(spec, corpus)
assert not isinstance(built, Unsupported)
model, data = built
print("parameters:", [p.name for p in model.parameters])
print("data columns:", model.data_columns)
print("likelihood:", model.likelihood.family, "| scale_expr set:", model.likelihood.scale_expr is not None)
print("marginal:", spec.marginal, "| rows:", data["outcome"].shape)

centered_model, centered_data = pool_model(spec.model_copy(update={"parametrization": "centered"}), corpus)
print("centered parameters:", [p.name for p in centered_model.parameters], "| data:", sorted(centered_data))

## `pool` → `Pooled` → `PoolResult`

`pool` returns a `Pooled` carrier: the `PoolResult` (a Spec of `ParameterSummary` objects, every
interval at the spec's `mass`/`definition`), the `Posterior` it was summarized from (with
`theta` draws), the model and the data. A backend that cannot stand behind its draws returns
`Unverified`; an unusable corpus returns `Unsupported`.

In [ ]:
out = pool(spec, corpus, backend="laplace", draws=1000, seed=0)
assert isinstance(out, Pooled) and isinstance(out.posterior, Posterior)
res: PoolResult = out.result
mu: ParameterSummary = res.mu
print(f"mu    {mu.mean:.3f} ± {mu.sd:.3f}  {mu.interval}   truth {MU}")
print(f"tau   {res.tau.mean:.3f} ± {res.tau.sd:.3f}  {res.tau.interval}   truth {TAU}")
print(f"gamma {res.gamma[0].name}: {res.gamma[0].mean:.3f} ± {res.gamma[0].sd:.3f}   truth {GAMMA}")
print("k =", res.k, "| effects =", res.n_effects, "| backend:", res.backend, "| delta:", res.delta)
print("detail:", res.detail["parametrization"])
print("round-trips:", Spec.from_json(res.to_json()) == res)

## Shrinkage against the analytic normal–normal factor

With `tau` fixed the model is linear-Gaussian and the Laplace posterior is exact up to Monte
Carlo: each effect's posterior mean is `(1 − B_g) ȳ_g + B_g mu` with
`B_g = se_g² / (se_g² + tau²)`. `EffectShrinkage` reports the `analytic` factor and the
`empirical` one read off the posterior means, `1 − (theta − mu) / (ȳ_g − mu)`. The empirical
ratio is ill-conditioned when a raw estimate sits close to `mu` (a small denominator amplifies
Monte Carlo noise), so it is compared only where `|ȳ_g − mu| > se_g`; every posterior mean is
checked against the analytic formula directly.

In [ ]:
fixed = PoolSpec(family="fertilizer", priors=PoolPriors(mu_scale=10.0, tau_fixed=TAU))
n_draws = 20_000  # Gaussian draws are cheap; MC error on a posterior mean is sd / sqrt(n)
out_fixed = pool(fixed, corpus, draws=n_draws, seed=1)
assert isinstance(out_fixed, Pooled)
fr = out_fixed.result
rows = fr.shrinkage
s0: EffectShrinkage = rows[0]
print(f"{'effect':8s} {'raw':>7s} {'se':>6s} {'theta':>7s} {'analytic B':>11s} {'empirical B':>12s}")
table(
    [
        [
            s.effect, f"{s.estimate:.3f}", f"{s.se:.3f}", f"{s.theta:.3f}", f"{s.analytic:.3f}",
            "(near mu)" if s.empirical is None or abs(s.estimate - fr.mu.mean) <= s.se
            else f"{s.empirical:.3f}",
        ]
        for s in rows[:6]
    ],
    headers=("effect", "estimate", "se", "theta", "analytic", "empirical"),
)

theta_mean = np.array([t.mean for t in fr.thetas])
B = np.array([s.analytic for s in rows])
predicted = (1 - B) * np.array([s.estimate for s in rows]) + B * fr.mu.mean
mc_sd = np.array([t.sd for t in fr.thetas]) / np.sqrt(n_draws)
print("\nposterior means within 4 MC sd of (1 − B) ȳ + B mu:", bool(np.all(np.abs(theta_mean - predicted) < 4 * mc_sd)))
well = [s for s in rows if s.empirical is not None and abs(s.estimate - fr.mu.mean) > s.se]
print(f"empirical vs analytic on {len(well)} well-conditioned effects: max relative gap",
      round(max(abs(s.empirical - s.analytic) / s.analytic for s in well), 4))
print("tau summary when fixed:", fr.tau.note, "| theta draws:", out_fixed.posterior.flat("theta_fertilizer").shape)

In [ ]:
order = np.argsort([s.estimate for s in rows])
fig = dumbbell(
    [f"{rows[i].effect}  (se {rows[i].se:.2f})" for i in order],
    [rows[i].estimate for i in order],
    [rows[i].theta for i in order],
    before_label="what the study reported", after_label="what the pool believes",
    title="Partial pooling, one study at a time",
    subtitle=f"tau fixed at {TAU} — each effect is pulled toward the family mean by its own noise",
    x_title="elasticity",
)
mark_x(fig, fr.mu.mean, text="family mean")
caption(fig, "The arrows are not the same length. A study with a small standard error barely "
             "moves; a noisy one is pulled most of the way to the mean. That is the whole "
             "content of the hierarchy, and the shrinkage factor B = se² / (se² + τ²) is the "
             "number behind each arrow.")

## Moderators: `moderator_matrix` and `ModeratorDesign`

The design is centered at the corpus mean so `mu` stays the family mean *at the average study*.
A record missing a moderator, or a moderator constant across the corpus, is a typed
`Unsupported` — never a silently imputed zero. `center` applies the same centering to a new
study's values.

In [ ]:
design = moderator_matrix(corpus, ["follow_up"])
assert isinstance(design, ModeratorDesign)
print(design.names, design.columns.shape, "| means:", design.means, "| column sums to 0:", round(float(design.column("follow_up").sum()), 10))
print("a new study at follow_up=10 is centered to", design.center({"follow_up": 10.0}))
missing = moderator_matrix(corpus, ["dose_level"])
print(type(missing).__name__, "-", missing.reason[:70], "...")

## The provenance bias `delta`: identified only by dual-read contributors

`delta_identification` is a `Verdict`: `identified` when at least one contributor in the family
reports both a model read and an experimental read, else `blocked` with the reason
`NO_DUAL_READ`. The exchangeability route — comparing model-only with experiment-only
contributors — is refused, because it would let *who runs experiments* masquerade as a
provenance bias.

Below, six contributors add a model read of the same effect with a true offset of `+0.4`.

In [ ]:
DELTA = 0.4
dual = tuple(
    StudyRecord(study=f"m{g}", contributor=f"c{g}", quantity="elasticity",
                estimate=float(theta_true[g] + GAMMA * (follow_up[g] - follow_up.mean()) + DELTA + 0.1 * rng.normal()),
                se=0.1, read="model", family="fertilizer", moderators={"follow_up": float(follow_up[g])})
    for g in range(6)
)
with_dual = Corpus(records=records + dual, name="with-dual-reads")
v_yes: Verdict = delta_identification(with_dual, "fertilizer")
v_no = delta_identification(corpus, "fertilizer")
print("with dual reads:   ", v_yes.status, "|", v_yes.route)
print("without dual reads:", v_no.status, "|", v_no.reason, "| matches NO_DUAL_READ:", NO_DUAL_READ in v_no.reason)

In [ ]:
bias_spec = PoolSpec(family="fertilizer", moderators=("follow_up",), bias_term=True,
                     priors=PoolPriors(mu_scale=2.0, delta_scale=1.0))
identified = pool(bias_spec, with_dual, draws=1000, seed=2)
assert isinstance(identified, Pooled)
d = identified.result.delta
print(f"identified: delta = {d.mean:.3f} ± {d.sd:.3f}  {d.interval}  identified={d.identified}   truth {DELTA}")
print("   mu:", round(identified.result.mu.mean, 3), "| verdict:", identified.result.delta_verdict.status)

not_identified = pool(bias_spec, corpus, draws=1000, seed=2)
assert isinstance(not_identified, Pooled)
d = not_identified.result.delta
print(f"\nnot identified: delta = {d.mean:.3f} ± {d.sd:.3f}  identified={d.identified}")
print("   posterior sd ≈ prior sd 1.0:", abs(d.sd - 1.0) < 0.15, "| note:", d.note)
print("   detail:", not_identified.result.detail["delta"][:60], "...")

In [ ]:
rng_d = np.random.default_rng(0)
with_dual_draws = rng_d.normal(identified.result.delta.mean, identified.result.delta.sd, 20_000)
without_draws = rng_d.normal(not_identified.result.delta.mean, not_identified.result.delta.sd, 20_000)
fig = density(
    {"with dual-read contributors": with_dual_draws, "without them": without_draws},
    colors=(BLUE, ORANGE),
    title="A parameter that is only estimable if somebody reported both",
    subtitle="posterior for the provenance offset delta, with and without contributors who did",
    x_title="delta",
)
mark_x(fig, DELTA, text="truth")
caption(fig, "The wide curve is the prior, handed back unchanged: without a contributor who "
             "reported both reads of the same effect there is nothing in the data about "
             "delta. The result says so — `identified=False`, with a note — rather than "
             "reporting the posterior mean as a finding.")

## Twelve studies, or three clients?

Every pool above treats its effects as exchangeable draws from one family. That is the right
model when the studies really are unrelated. It is the wrong one for a house running repeated
experiments for the same few clients: four studies in one client's markets, run by the same
team on the same seasonality, are not four independent draws about the world.

The cost lands on `mu`. `effect_key="study"` gives every record its own effect and estimates
one `tau`, so the family mean is credited with twelve independent units. `effect_key="nested"`
puts a party level above the studies and splits the variance in two:

    alpha_p ~ N(mu, tau_party)                     p = 1..P    parties
    theta_g ~ N(alpha_p(g), tau)                   g = 1..G    studies within a party
    y_i     ~ N(theta_g(i) + ..., se_i)

Simulate exactly that world — three parties whose true effects differ by `tau_party = 0.4`,
four studies each differing within a party by `tau = 0.15` — and ask both models.

In [ ]:
MU_N, TAU_PARTY, TAU_STUDY = 0.5, 0.4, 0.15
rng_n = np.random.default_rng(0)
parties = ["northwind", "acme", "globex"]
alpha_true = {p: MU_N + TAU_PARTY * o for p, o in zip(parties, (-1.0, 0.2, 0.9))}

nested_records = []
for party in parties:
    for j in range(4):
        theta = alpha_true[party] + TAU_STUDY * rng_n.standard_normal()
        se = float(rng_n.uniform(0.1, 0.2))
        nested_records.append(
            StudyRecord(study=f"{party}-{j}", contributor=party, quantity="elasticity",
                        family="fert", estimate=float(theta + se * rng_n.standard_normal()),
                        se=se, read="experiment")
        )
book = Corpus(records=tuple(nested_records), name="the book of business")
print(len(book), "records |", len(book.contributors()), "parties")
print("true party effects:", {k: round(v, 3) for k, v in alpha_true.items()}, "| true mu", MU_N)

### The same data, two beliefs about what is independent

`tau_party` is fixed here at its true value, which is what lets `laplace` fit the tree exactly
— the party level is non-centered, so with `tau_party` free the likelihood sees only the
product `tau_party · z_p` and a Laplace expansion at one point of that ridge overstates it
threefold. `pool` refuses that combination rather than returning the number; a sampler
backend estimates it.

In [ ]:
rows = []
fits = {}
for key, priors in (("study", PoolPriors()), ("contributor", PoolPriors()),
                    ("nested", PoolPriors(tau_party_fixed=TAU_PARTY))):
    fit = pool(PoolSpec(family="fert", effect_key=key, priors=priors),
               book, backend="laplace", draws=4000, seed=1)
    assert isinstance(fit, Pooled)
    fits[key] = r = fit.result
    rows.append([key, f"{r.n_effects} effects / {r.n_parties or '—'} parties",
                 f"{r.mu.mean:.3f}", r.mu.interval.text(), f"{r.mu.interval.width:.3f}",
                 f"{r.tau.mean:.3f}", "—" if r.tau_party is None else f"{r.tau_party.mean:.3f}"])
table(rows, headers=("effect_key", "levels", "mu", "interval on mu", "width", "tau", "tau_party"),
      title=f"the same twelve records, three beliefs about them (true mu = {MU_N})")

refused = pool(PoolSpec(family="fert", effect_key="nested"), book, backend="laplace")
print("nested with a free tau_party under laplace:", type(refused).__name__)
print(" ", refused.reason[:150], "...")

The `study` pool reports the family mean to within a narrow interval, and it is not entitled
to: it counted twelve markets where there are three clients. The nested pool's interval on `mu`
is more than twice as wide, and its `tau` — now the spread *within* a client — is close to the
0.15 it was simulated with, instead of the mixture of within and between the flat pool
reports.

`contributor` is the honest half-measure. It gets the count right — three effects, three
clients — and its interval on `mu` sits between the other two, but it buys that by asserting
that a client's four studies all estimate one effect, so its `tau` is again a mixture and there
is no within-client spread to report. The nested tree is the one that gets both.

The level also changes what each study is shrunk toward. Under the flat pools an outlying study
is pulled toward the family mean; under the nested one it is pulled toward *its own client's*
mean, which is where the information about it actually is.

In [ ]:
nested_result = fits["nested"]
table(
    [[a.name.split("[")[1][:-1], f"{a.mean:.3f}", a.interval.text(), f"{alpha_true[a.name.split('[')[1][:-1]]:.3f}"]
     for a in nested_result.alphas],
    headers=("party", "alpha", "interval", "truth"), title="what each party's mean came out at",
)

table(
    [[row.effect, row.party, f"{row.estimate:+.3f}", f"{row.theta:+.3f}", f"{row.toward:+.3f}",
      f"{row.analytic:.2f}"] for row in nested_result.shrinkage[:6]],
    headers=("study", "party", "raw", "shrunk", "toward", "shrinkage"),
    title="the first six studies: each pulled toward its own party, not toward mu",
)
print("family mean mu =", round(nested_result.mu.mean, 3), "— no study is shrunk toward it directly")
print("delta stays available here:", PoolSpec(family="fert", effect_key="nested", bias_term=True).bias_term,
      "— a party's two reads still share its alpha")

## What this bought you

Partial pooling that shrinks each study by its own precision, a between-study sd estimated
rather than assumed, moderators centred so the family mean stays interpretable, and a
provenance offset that comes back as a refusal when the corpus cannot identify it.

And, when the studies came from a handful of parties rather than from the world, a second
variance component that stops the family mean claiming a precision the book of business does
not contain — while keeping `delta` identified, which `effect_key="study"` cannot do.

`03-priors-handoff.ipynb` hands the pooled posterior forward as the next study's prior.